# Chapter 16: Least-Squares Regression

**MAD 2502 — Computational Mathematics with Python**

Regression uses a mathematical model to describe a relationship in data. At this level, the goal is to understand the model, residuals, and evidence—not to chase the most complicated model.

**Learning rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect.

## Learning goals

You should be able to define residuals, explain least squares, fit a straight line, interpret slope/intercept, compute and interpret $R^2$ with appropriate caveats, inspect residuals, and fit a simple nonlinear model with SciPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

rng = np.random.default_rng(2502)

## Understand: model, prediction, residual

Suppose a model predicts $\hat y_i$ for an observed value $y_i$. The **residual** is

$$r_i = y_i - \hat y_i.$$

Least-squares regression chooses model parameters to minimize the **sum of squared errors**

$$SSE = \sum_i r_i^2.$$

Squaring prevents positive and negative residuals from canceling and penalizes large misses strongly.

## Predict → Run: fitting a line

We will model $y \approx mx+b$. Before fitting, look at the scatter plot and predict whether the slope should be positive or negative.

In [ ]:
x = np.linspace(0, 10, 25)
y = 2.5 * x + 4 + rng.normal(0, 2.5, size=x.size)

fig, ax = plt.subplots()
ax.scatter(x, y)
ax.set(xlabel='x', ylabel='y', title='Data before fitting')
plt.show()

In [ ]:
slope, intercept = np.polyfit(x, y, 1)
predicted = slope * x + intercept
residuals = y - predicted
sse = np.sum(residuals**2)

print('slope:', slope)
print('intercept:', intercept)
print('SSE:', sse)

The slope estimates how much the model predicts `y` changes for a one-unit increase in `x`. The intercept is the model's predicted `y` when `x=0`; it is only meaningful if `x=0` makes sense in context.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x, y, label='data')
ax.plot(x, predicted, linestyle='--', label='least-squares line')
ax.set(xlabel='x', ylabel='y', title='Linear least-squares fit')
ax.legend()
plt.show()

## $R^2$ for a model with an intercept

For an ordinary regression model that includes an intercept,

$$R^2 = 1 - \frac{\sum (y_i-\hat y_i)^2}{\sum (y_i-\bar y)^2}.$$

It measures the fraction of variation around the mean accounted for by the fitted model. A high $R^2$ does **not** prove causation or guarantee that the model is appropriate.

In [ ]:
sst = np.sum((y - y.mean())**2)
r_squared = 1 - sse / sst
print('R^2:', r_squared)

## Residual plots

A useful model should leave residuals without an obvious systematic pattern. Structure in the residual plot is evidence that the model may be missing something.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x, residuals)
ax.axhline(0, linestyle='--')
ax.set(xlabel='x', ylabel='residual', title='Residual plot')
plt.show()

## Change

Increase the noise in the generated data. Predict what will happen to the scatter around the line, SSE, and $R^2$. Then decrease the noise and compare.

## Through-origin models

Sometimes theory requires a model $y \approx ax$ with no intercept. Do not remove the intercept merely to improve a story; it should be justified by the problem. Standard $R^2$ interpretations also require care for no-intercept models.

In [ ]:
a = np.sum(x * y) / np.sum(x**2)
print('through-origin coefficient:', a)

## Use: nonlinear fitting with `curve_fit`

SciPy can estimate parameters in a model you define. Here the model is exponential.

In [ ]:
def exponential_model(x, a, b):
    return a * np.exp(b * x)

x2 = np.linspace(0, 3, 25)
y2 = exponential_model(x2, 2.0, 0.7) + rng.normal(0, 0.8, size=x2.size)

parameters, covariance = curve_fit(exponential_model, x2, y2, p0=(1, 0.5))
a_fit, b_fit = parameters
print(a_fit, b_fit)

fig, ax = plt.subplots()
ax.scatter(x2, y2, label='data')
ax.plot(x2, exponential_model(x2, a_fit, b_fit), linestyle='--', label='fitted exponential')
ax.set(xlabel='x', ylabel='y', title='Nonlinear least-squares fit')
ax.legend()
plt.show()

## Polynomial fitting and overfitting

Higher-degree polynomials can bend more, but extra flexibility can fit noise rather than meaningful structure. A more complicated model is not automatically a better scientific explanation.

In [ ]:
quadratic_coefficients = np.polyfit(x, y, 2)
quadratic = np.polyval(quadratic_coefficients, x)

fig, ax = plt.subplots()
ax.scatter(x, y, label='data')
ax.plot(x, predicted, label='linear')
ax.plot(x, quadratic, linestyle='--', label='quadratic')
ax.legend()
ax.set(title='More flexible does not always mean more useful')
plt.show()

## Practice

1. Fit a line to a small dataset and report slope/intercept.
2. Compute residuals and SSE.
3. Plot the residuals and describe any pattern.
4. Explain one reason a high $R^2$ could still be misleading.
5. State when a through-origin model might be justified.
6. Compare a linear and quadratic model using both the plot and the meaning of the problem.

## Connect to the capstone

Regression can support a strong beginner capstone when the question and variables justify it. Report what the variables mean, show the data and fitted model, inspect residuals, state assumptions/caveats, and avoid causal language unless the study design truly supports causal inference.